# Lab 1 – K-Nearest Neighbours

In this lab session, we will implement the K-Nearest Neighbours (KNN) algorithm.  

Specifically, we will work through a typical machine learning pipeline, which involves:

1. Defining the problem we want to solve
2. Collecting the data
3. Preprocessing the data
4. Define the variables of the problem
5. Implementing a KNN model
6. Use the model to make predictions on unseen data

#### Lab Materials

1. The KNN slides available on ILIAS.
2. For an introduction to Python and useful libraries, we highly recommend reviewing the chapter **“2.3 Lab: Introduction to Python”** from the book *An Introduction to Statistical Learning*.

The book is available [here](https://www.statlearning.com/).

# Step 1&2: Defining the Problem

In this lab, we will use the data collected during the course to create a model that predicts the use of AI based on the available variables.

We begin by loading the data into a Pandas DataFrame.


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("class.csv")

In [ ]:
# First, we rename the columns for convenience.
rename_dict = {
               'grade %': 'grade',
               'sport (hours)':'sport',
               'drugs (Alc/Coffee)':'drugs',
               'AI use [0, 1, 2, 3]: 3 means it solves everything, 1 means I use it when stuck, and 2 is: I verify what it does':"AI_ussage"
              }
data.rename(columns= rename_dict,inplace =True)

In [ ]:
data

Because the labels take only two possible values, 1 and 2, we can treat this as a binary classification problem.

For convenience, we relabel the two classes as 0 and 1, where 0 indicates low AI usage and 1 indicates high AI usage.

In [ ]:
mapping = {
    1.0: 0,
    2.0: 1
}

data['AI_ussage'] = data['AI_ussage'].map(mapping)

# Step 3: Data Preprocessing

As in many real-world applications, the data contain inconsistencies and require preprocessing. 

There is no single correct way to preprocess a dataset, and the choices made during preprocessing can significantly affect the algorithm’s performance.

For example, we can observe that the `drugs` column contains categorical values, which must be encoded numerically before they can be used by the KNN algorithm.

In [ ]:
# Example of label encoding a categorical column
mapping = {
    'Coffee': 1,
    'Alc': 2,
    'Alc+Coffee': 3
}

data['drugs'] = data['drugs'].map(mapping)

In [ ]:
data

Before implementing KNN, examine the dataset and answer the following questions:

1. What problems or inconsistencies do you observe in the data?
2. What preprocessing steps did you apply?
3. How might these choices affect the performance of the classifier?

#### Basic information about the dataset

Before starting a machine learning project, it is important to explore the dataset. 
    
This helps us understand the problem and identify issues, such as missing values or inconsistent entries, that may need to be addressed before training a model.

In [ ]:
print("Shape of our dataset:", data.shape)
print("Number of features:", data.shape[1])
print("Number of examples:", data.shape[0])

In [ ]:
label = "AI_ussage"
print("Number of different classes", len(data[label].unique()))
print("unique classes:", data[label].unique())

In [ ]:
data[label].value_counts()

#### Study the features

The dataset contains several features.
Some may not be useful for predicting the target, such as identifiers.  
Studying the features helps us understand the problem and develop intuition about what information may be useful for prediction.  
It can also reveal that important information is missing.   
If the available features do not capture what we need, we may need to collect additional data or find a more suitable dataset.  

For example, our dataset contains students’ names, which are unlikely to be informative for this prediction task. We will therefore exclude them from the analysis.

In [ ]:
data.columns

In [ ]:
active_features=['grade', 'device', 'sport', 'drugs', 'AI_ussage']

#### Missing data

Our dataset also contains missing values.  
A simple way to handle them is to remove rows with missing information.   
However, this reduces the number of training samples and may affect the model’s performance.  

Another option is to fill in missing values.   
For example, we could replace a missing numerical value with the average calculated from the training data, or use a model to estimate it from available features.   
These choices can also affect performance, so compare them with the simple approach of removing incomplete rows.

The [scikit-learn documentation on imputing missing values](https://scikit-learn.org/stable/modules/impute.html) describes several methods for filling in missing data, including mean imputation and K-nearest-neighbours imputation.

For the perpuse of the current lab, we just remove the missing rows with missing values.

In [ ]:
data[active_features].isna().mean()

I will exclude the **“drugs”** feature from this analysis. About 32% of participants did not answer this question, so removing every row with a missing value in this feature would discard a large part of the dataset. The other features have relatively few missing values, so I will remove rows that are missing those values.

The “drugs” question also asks for sensitive personal information, which some participants may prefer not to share.   
We may discuss ways to protect participants’ privacy later in the course.

In [ ]:
data.isna().mean()

In [ ]:
# active feature
active_features = ['grade', 'device', 'sport', 'AI_ussage']
mask  = data[active_features].isna().any(axis=1)
data = data[active_features][np.logical_not(mask)]

In [ ]:
data.isna().mean()

### Visualisation and descriptive statistics

Plots and basic statistics help us become familiar with the data.   
In particular, we should examine the range and scale of each numerical feature, since these can affect the performance of some algorithms.

For example, K-nearest neighbours (KNN) uses a distance measure to find similar samples.   
If features have very different numerical scales, the feature with the larger range may dominate the distance calculation. Scaling the features can help prevent this.

In [ ]:
data.describe()

In [ ]:
plt.figure()
data.plot.density(subplots=True,
                  layout=(5, 1),
                  figsize=(8, 12),
                  sharex=True,
                  sharey=False)
plt.show()
plt.close()

### Data scaling

A common way to scale numerical features is **standard scaling**. 

For each feature, we subtract its mean and divide by its standard deviation:

$$
x_{\text{scaled}} = \frac{x - \mu}{\sigma},
$$

where $\mu$ is the feature’s mean and $\sigma$ is its standard deviation. 

After scaling, the feature has a mean of approximately 0 and a standard deviation of 1 on the training data.

In [ ]:
features = ["grade", "device", "sport"]
mean_data = data[features].mean()
std_data = data[features].mean()

data[features] =  (data[features]-mean_data) / std_data

In [ ]:
plt.figure()
data.plot.density(subplots=True,
                  layout=(5, 1),
                  figsize=(8, 12),
                  sharex=True,
                  sharey=False)
plt.show()
plt.close()

In [ ]:
data

# Step 4: Define the Problem Variables

Which variables will you use to predict AI_usage?

In [ ]:
# Define variables
label = "AI_ussage"
features = ["grade", "device", "sport"] # fill this.

So after preprocessing, we have a binary classification problem with three features.

# Step 5: Implement the KNN Algorithm

Below, we provide a template for the KNN model that you must complete.

More specifically, you should:

1. Implement the `get_probabilities(self, x)` method, which estimates the probability of each class for a test sample `x`.
2. Implement the `predict` method, which returns the class label with the highest estimated probability.

In [ ]:
def euclidean_metric(x, y):
    return np.linalg.norm(x - y)


## Skeleton code to be filled in
class NearestNeighbourClassifier:
    ## Initialise the neighbours with a specific metric function and dataset
    ## Assume labels are in {1, ..., m}
    def __init__(self, data, labels, metric, K):
        self.metric = metric
        self.data = data
        self.labels = labels
        self.n_classes = len(np.unique(labels))  # Counts actual number of labels
        self.K = K
        self.n_points = data.shape[0] # np array dimensions
        self.n_features = data.shape[1]
        print("classes: ", self.n_classes)

    # Gives a utility for every possible choice made by the algorithm
    def decide(self, U, x):
        """
        A method that return the action that maximise the expected utility.
        :param U: is a 2 denominational array that indicated the utility of each action a given y
                    example: U = np.array([ [ 1 , -1000],
                                            [ -1 ,    0]  ])
                            so U[1,0]=-1 is the utility of taking action a=1 when y=0.
        :param x: the test point.
        :return: the action that maximises the expected utility max_a E[U|a,x].
                 where E[U|a,x] = sum_y P(y|x) U(a,y).
        """
        # HINT:
        # Need to use the get_probabilities function to return the action with the highest
        # expected utility
        # i.e. maximising sum_y P(y|x) U(a,y)
        p_y_x = self.get_probabilities(x)
        E_U_a_0 = sum(U[0,:] * p_y_x)
        E_U_a_1 = sum(U[1,:] * p_y_x)
        E_U = np.array([E_U_a_0, E_U_a_1])
        a = np.argmax(E_U)
        return a
    
    ## predict the most likely label
    def predict(self, x):
        # calculate the probabilities of different clases
        p = self.get_probabilities(x)
        # return the y value for the closest point
        return np.argmax(p)
    

    ## return a vector of probabilities, one for each label
    ## Each component of the vector corresponds to the ratio of that same label in the set of neighbours
    def get_probabilities(self, x):
        # 1. calculate distances
        distances = [self.metric(x, self.data[t]) for t in range(self.n_points)]
        # 2. sort data using argsort
        # 3. get K closest neighbours
        neighbours = np.argsort(distances)[0:self.K]
        # 4. get the proportion of each label so that proportions[y] is the proportion of label y in the neighbourhood
        proportions = np.zeros(self.n_classes)
        for k in range(self.K):
            label = int(self.labels[neighbours[k]])
            proportions[label] += 1
        proportions /= self.K
        return proportions

In [ ]:
kNN_k1 = NearestNeighbourClassifier(data = data[active_features].values, 
                                 labels= data[label].values,
                                 K =  1,
                                 metric = euclidean_metric)

In [ ]:
#### check the get probabilities and the predictions

In [ ]:
x = data[active_features].values[4]

In [ ]:
kNN_k1.get_probabilities(x)

In [ ]:
kNN_k1.predict(x)

##### Question
After implementing your algorithm, evaluate its accuracy on the entire training set using \(k=1\) and \(k=5\).
What do you observe?

In [ ]:
def accuracy(true_labels, predictions):
    acc = true_labels == predictions
    acc = acc.mean()
    return acc

In [ ]:
### accuracy for k=1

In [ ]:
kNN_k1 = NearestNeighbourClassifier(data = data[active_features].values, 
                                 labels= data[label].values,
                                 K =  1,
                                 metric = euclidean_metric)

In [ ]:
predictions = []
for data_point in data[active_features].values:
    pred = kNN_k1.predict(data_point)
    predictions += [pred] 
predictions = np.array(predictions)
predictions

In [ ]:
accuracy(true_labels = data[label].values, predictions = predictions)

### accuracy for k=5

In [ ]:
kNN_k5 = NearestNeighbourClassifier(data = data[active_features].values, 
                                    labels= data[label].values,
                                    K =  5,
                                    metric = euclidean_metric)


In [ ]:
predictions = []
for data_point in data[active_features].values:
    pred = kNN_k5.predict(data_point)
    predictions += [pred] 
predictions = np.array(predictions)
predictions

In [ ]:
accuracy(true_labels = data[label].values, predictions = predictions)

# Step 6: Utility-Based Decisions

We will now use a utility function to make predictions. 
More specifically, for each data point, we will select the action that maximizes the expected utility.

The utility matrix $U$ is a two-dimensional array in which $U[a,y]$ represents the utility of taking action $a$ when the true class is $y$.

For example:

```python
U = np.array([
    [ 1, -1000],
    [-1,     0]
])
```

In this example:

* $U[0,0]=1$: the utility of selecting action (or predict) $0$ when the true class is $0$.
* $U[0,1]=-1000$: the utility of selecting action $0$ when the true class is $1$.
* $U[1,0]=-1$: the utility of selecting action $1$ when the true class is $0$.
* $U[1,1]=0$: the utility of selecting action $1$ when the true class is $1$.


#### tasks
1) First, implement the `decide` method.
2) Then, use each of the two utility functions below (U_1 and U_2) to make predictions with KNN for \(k=3\), and calculate the resulting accuracy.

What do you observe? How does the accuracy obtained with each utility function compare with the accuracy of the standard KNN classifier?


### A. Expected utility

First, we interpret the model’s output as estimated probabilities for the two classes, $P(Y=0\mid x)$ and $P(Y=1\mid x)$. We can then calculate the expected utility of each possible decision, $a=0$ or $a=1$.

For decision $a=0$:

$$
\mathbb{E}_{Y\sim P(\cdot\mid x)}[U(0,Y)]
=
U(0,0)P(Y=0\mid x)
+
U(0,1)P(Y=1\mid x).
$$

Similarly, for decision $a=1$:

$$
\mathbb{E}_{Y\sim P(\cdot\mid x)}[U(1,Y)]
=
U(1,0)P(Y=0\mid x)
+
U(1,1)P(Y=1\mid x).
$$

We choose the decision with the higher expected utility.

In [ ]:
U_2 = np.array([[1, -1000],
                [-1, 1]])

p_y_x = [0.7, 0.3] # lets say that for a given x we have P(y|x)
Expected_utility_a_0 = U_2[0,0] * p_y_x[0] + U_2[0,1] * p_y_x[1] # expected utility for predicting the class a=0
Expected_utility_a_1 = U_2[1,0] * p_y_x[0] + U_2[1,1] * p_y_x[1] # expected utility for predicting the class a=0

In [ ]:
sum(U_2[0,:] * p_y_x)

In [ ]:
Expected_utility_a_0

In [ ]:
Expected_utility_a_1

In this expample while the most probable class is y=0 we will output a=1 as it has better utility.

### B. Testing different utilities

With $U_1$, choosing the action with the highest expected utility is the same as choosing the most probable class: correct decisions have utility 1, and incorrect decisions have utility 0 and thus:

$$
\mathbb{E}[U_1(a,Y)\mid x] = P(Y=a\mid x).
$$

With $U_2$, mistakes have different costs. Choosing $a=0$ when the true class is $1$ has utility $-1000$, while choosing $a=1$ when the true class is $0$ has utility $-1$. As a result, the model may choose $a=1$ even when class $0$ is more probable. This does not mean it will always choose $a=1$; the decision still depends on the estimated class probabilities.

In [ ]:
U_1 = np.array([[1, 0],
                [0, 1]])

In [ ]:
kNN_k5 = NearestNeighbourClassifier(data = data[active_features].values, 
                                    labels= data[label].values,
                                    K =  5,
                                    metric = euclidean_metric)

In [ ]:
predictions_u1 = []
for data_point in data[active_features].values:
    pred = kNN_k5.decide(U=U_1, x = data_point)
    predictions_u1 += [pred] 
predictions_u1 = np.array(predictions_u1)
predictions_u1

In [ ]:
predictions_u1

In [ ]:
predictions ==  predictions_u1

In [ ]:
predictions_u2 = []
for data_point in data[active_features].values:
    pred = kNN_k5.decide(U=U_2, x = data_point)
    predictions_u2 += [pred] 
predictions_u2 = np.array(predictions_u2)
predictions_u2

In [ ]:
predictions ==  predictions_u2

In [ ]:
predictions_u1 == predictions_u2